In [0]:
%run /Workspace/Users/pramotha@systechusa.com/qa-validation-framework/config/validation_queries_and_logs

In [0]:
import pandas as pd

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE project_training_amh.default.qa_validation_results (
    check_name STRING,
    check_category STRING,
    status STRING,
    source_table STRING,
    target_table STRING,
    key_columns STRING,
    compared_columns STRING,
    src_count INT,
    tgt_count INT,
    mismatch_count INT,
    src_only_count INT,
    tgt_only_count INT,
    value_mismatch_count INT,
    duplicate_count INT,
    null_count INT,
    execution_time_ms INT,
    remarks STRING,
    run_id STRING,
    timestamp TIMESTAMP
)
""")

In [0]:
src_table = "project_training_amh.default.employee_src_qa"
tgt_table = "project_training_amh.default.employee_tgt_qa"

In [0]:
results = []

# Row Count
results.append(row_count_check(spark, src_table, tgt_table))

# Null Check
# results.append(null_check(spark, src_table))

# Duplicate Check
results.append(duplicate_check(spark, tgt_table, "emp_id"))

# GroupBy Check
results.append(groupby_check(spark, src_table, tgt_table, ["dept_id"]))

# Referential Check
results.append(referential_check(spark, tgt_table, src_table, "emp_id"))

# Column-level comparison for all records using key columns
results.append(column_level_comparison_check(spark, src_table, tgt_table, ["emp_id"]))

In [0]:
display(pd.DataFrame(results))

In [0]:
log_results(spark, results)

In [0]:
%sql
select * from project_training_amh.default.qa_validation_results

In [0]:
%skip
data = [
(1,"Arun",101,50000,"arun@systech.com","2026-04-20",True),
(2,"Bala",102,60000,"bala@systech.com","2026-04-20",True),
(3,"Chitra",101,55000,"chitra@systech.com","2026-04-20",True),
(4,"Deepak",103,70000,"deepak@systech.com","2026-04-21",True),
(5,"Esha",104,65000,"esha@systech.com","2026-04-21",True),
(6,"Farhan",102,48000,"farhan@systech.com","2026-04-21",True),
(7,"Gita",101,52000,"gita@systech.com","2026-04-21",True),
(8,"Hari",103,72000,"hari@systech.com","2026-04-21",True),
(9,"Isha",104,58000,"isha@systech.com","2026-04-22",True),
(10,"John",102,61000,"john@systech.com","2026-04-22",True),
(11,"Kiran",101,53000,"kiran@systech.com","2026-04-22",True),
(12,"Latha",104,67000,"latha@systech.com","2026-04-22",True),
(13,"Manoj",103,75000,"manoj@systech.com","2026-04-22",True),
(14,"Nisha",102,62000,"nisha@systech.com","2026-04-22",True),
(15,"Omkar",101,54000,"omkar@systech.com","2026-04-23",True),
(16,"Pooja",104,69000,"pooja@systech.com","2026-04-23",True),
(17,"Qadir",103,71000,"qadir@systech.com","2026-04-23",True),
(18,"Ravi",102,60000,"ravi@systech.com","2026-04-23",True),
(19,"Sneha",101,56000,"sneha@systech.com","2026-04-23",True),
(20,"Tarun",104,68000,"tarun@systech.com","2026-04-23",True),
(21,"Uma",105,64000,"uma@systech.com","2026-04-24",True),
(22,"Vikram",106,73000,"vikram@systech.com","2026-04-24",False)
]

cols = ["emp_id","emp_name","dept_id","salary","email","last_updated","is_active"]

spark.createDataFrame(data, cols) \
    .write.mode("overwrite") \
    .saveAsTable("project_training_amh.default.employee_src_qa")

In [0]:
%skip
data = [
(1,"Arun",101,50000,"arun@systech.com","2026-04-20",True),
(2,"Bala",102,60000,"bala@systech.com","2026-04-20",True),
(3,"Chitra",101,55000,"chitra@systech.com","2026-04-20",True),
(4,"Deepak",103,70000,"deepak@systech.com","2026-04-21",True),
(5,"Esha",104,None,"esha@systech.com","2026-04-21",True),        # NULL issue
(6,"Farhan",102,48000,"farhan@systech.com","2026-04-21",True),
(7,"Gita",101,52000,"gita@systech.com","2026-04-21",True),
(8,"Hari",103,72000,"hari@systech.com","2026-04-21",True),
(9,"Isha",104,58000,"isha@systech.com","2026-04-22",True),
(10,"John",102,61000,"john@systech.com","2026-04-22",True),
(11,"Kiran",101,53000,"kiran@systech.com","2026-04-22",True),
(12,"Latha",104,67000,"latha@systech.com","2026-04-22",True),
(13,"Manoj",103,75000,"manoj@systech.com","2026-04-22",True),
(14,"Nisha",102,62000,"nisha@systech.com","2026-04-22",True),
(15,"Omkar",101,99999,"omkar_updated@systech.com","2026-04-23",True),     # Value mismatch
(16,"Pooja",104,69000,"pooja@systech.com","2026-04-23",True),
(17,"Qadir",103,71000,"qadir@systech.com","2026-04-23",True),
(18,"Ravi",102,60000,"ravi@systech.com","2026-04-23",True),
(19,"Sneha",101,56000,"sneha@systech.com","2026-04-23",True),
(19,"Sneha",101,56000,"sneha@systech.com","2026-04-23",True),      # Duplicate
(21,"Uma",105,64000,"uma@systech.com","2026-04-24",False),         # is_active mismatch
(23,"Yash",107,70000,"yash@systech.com","2026-04-24",True)         # Extra record in target
# Missing emp_id = 20 and 22
]

cols = ["emp_id","emp_name","dept_id","salary","email","last_updated","is_active"]

spark.createDataFrame(data, cols) \
    .write.mode("overwrite") \
    .saveAsTable("project_training_amh.default.employee_tgt_qa")